# Statistical Concepts in Quantitative Time Series Management

A comprehensive guide to time series analysis, forecasting, and risk management in quantitative finance. This notebook covers stationarity, autocorrelation, ARIMA/GARCH models, mean reversion, and practical trading applications.

## 1. Import Required Libraries

Load essential libraries for time series analysis, statistical testing, and visualization.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, jarque_bera, norm
from statsmodels.tsa.stattools import adfuller, acf, pacf, kpss, arima_order_select_ic
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
print("✓ All libraries imported successfully")

## 2. Generate Mock Time Series Data

Create synthetic time series data with different characteristics: trend, seasonality, stationarity, and mean reversion.

### Why Do We Need Diverse Mock Time Series Types?

**The Core Problem:** A single random walk cannot teach you to distinguish between fundamentally different market regimes. Real quant work requires understanding which regime you're in before choosing any model.

**Practical Examples:**
- **Trend-following vs. Mean-Reversion Strategies**: Trend-following CTAs (Commodity Trading Advisors like AHL, Winton) profit when a series has persistent trend. Mean-reversion funds (Renaissance, Two Sigma stat-arb desks) profit when the series reverts. Generating both types lets you verify your regime-detection algorithm before deploying in live markets.
- **Seasonal Adjustment in Fixed Income**: Bond yields have macroeconomic seasonal patterns (year-end balance sheet effects, tax flows). Generating synthetic seasonal series lets you test if your model correctly separates seasonal patterns from real rate changes.
- **ARIMA-Type Processes in FX Markets**: Exchange rates exhibit autocorrelation during trending macro regimes. Generating ARIMA(1,1,1) mock data tests whether your model order-selection correctly identifies these dependencies.
- **Regime Switching for Portfolio Allocation**: A multi-asset portfolio model needs to identify whether each asset is currently in a trend or mean-reversion regime to apply the correct signal. Mock data with known properties validates regime-detection accuracy before deploying on opaque real data.

**The Four Archetypes:**
| Series Type | Financial Analogy | Typical Market |
|------------|------------------|----------------|
| Trend + noise | Tech stock bull market | 2010–2021 US equities |
| Mean-reverting (OU) | Spread between pairs | FX, bond spreads |
| Seasonal | Agricultural commodity | Natural gas, wheat |
| ARIMA(1,1,1) | Macro-driven FX rate | EUR/USD, JPY/USD |

In [ ]:
# Generate time periods
n_periods = 500
dates = pd.date_range(start='2022-01-01', periods=n_periods, freq='D')

# 1. Trend + Noise (Non-stationary)
trend = np.linspace(100, 150, n_periods)
noise_trend = trend + np.random.normal(0, 2, n_periods)

# 2. Mean-reverting series (Stationary)
mean_reversion = 100
reversion_speed = 0.05
mean_revert_series = np.zeros(n_periods)
mean_revert_series[0] = 100
for t in range(1, n_periods):
    mean_revert_series[t] = mean_revert_series[t-1] + reversion_speed * (mean_reversion - mean_revert_series[t-1]) + np.random.normal(0, 1)

# 3. Seasonal pattern
seasonal = 100 + 10 * np.sin(2 * np.pi * np.arange(n_periods) / 252) + np.random.normal(0, 1, n_periods)

# 4. ARIMA(1,1,1) process
ar_coef = 0.6
ma_coef = 0.3
arima_series = np.zeros(n_periods)
residuals = np.random.normal(0, 2, n_periods)
for t in range(1, n_periods):
    arima_series[t] = ar_coef * arima_series[t-1] + residuals[t] + ma_coef * residuals[t-1]

# Create DataFrame
df_ts = pd.DataFrame({
    'trend_series': noise_trend,
    'mean_revert': mean_revert_series,
    'seasonal': seasonal,
    'arima': arima_series + 100
}, index=dates)

print("Mock Time Series Data Generated:")
print(df_ts.head(10))
print(f"\nShape: {df_ts.shape}")

# Plot all series
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

df_ts['trend_series'].plot(ax=axes[0, 0], color='steelblue', linewidth=1.5)
axes[0, 0].set_title('1. Trend + Noise (Non-Stationary)', fontweight='bold')
axes[0, 0].set_ylabel('Price')
axes[0, 0].grid(alpha=0.3)

df_ts['mean_revert'].plot(ax=axes[0, 1], color='green', linewidth=1.5)
axes[0, 1].axhline(mean_reversion, color='red', linestyle='--', alpha=0.7, label='Mean')
axes[0, 1].set_title('2. Mean-Reverting Series (Stationary)', fontweight='bold')
axes[0, 1].set_ylabel('Price')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

df_ts['seasonal'].plot(ax=axes[1, 0], color='orange', linewidth=1.5)
axes[1, 0].set_title('3. Seasonal Pattern', fontweight='bold')
axes[1, 0].set_ylabel('Price')
axes[1, 0].grid(alpha=0.3)

df_ts['arima'].plot(ax=axes[1, 1], color='purple', linewidth=1.5)
axes[1, 1].set_title('4. ARIMA(1,1,1) Process', fontweight='bold')
axes[1, 1].set_ylabel('Price')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Stationarity Testing (ADF & KPSS)

Test whether time series are stationary using Augmented Dickey-Fuller (ADF) and KPSS tests. Stationarity is crucial for time series modeling.

### Why Do We Need Stationarity Testing?

**The Core Problem:** Most time series models (ARIMA, VAR, OLS regression) assume the statistical properties of the data — mean, variance, autocorrelation — are constant over time. Non-stationary data violates this assumption, making model estimates meaningless and forecasts unreliable.

**Practical Examples:**
- **Spurious Regression (The Classic Trap)**: Regress the S&P 500 index level against US GDP level. You'll get R² > 0.95 and a highly significant t-statistic — but the relationship is entirely spurious because both are non-stationary random walks trending upward. The ADF test catches this before you build an invalid model.
- **Pair Trading Pre-Check**: Before entering a pairs trade on two bank stocks, a quant runs ADF on the spread. If the spread is non-stationary (unit root), the strategy will have unlimited drawdown because the spread can drift forever. Only stationary spreads mean-revert reliably.
- **Federal Reserve Research**: When analyzing the relationship between monetary base and inflation, Fed economists first test both series for integration order (how many differences needed), then apply error-correction models (VECM) only to co-integrated pairs. Using ARIMA on a non-stationary series produces wrong standard errors and invalid policy conclusions.
- **Algorithmic Signal Decay**: A momentum signal built on price levels (non-stationary) will generate very different behavior than a signal on returns (stationary). Stationarity testing determines whether to model levels or differences.

**ADF vs. KPSS — Why Use Both?**
| Test | Null Hypothesis | Reject When |
|------|----------------|-------------|
| ADF | Unit root (non-stationary) | p < 0.05 → Stationary |
| KPSS | Stationary | p < 0.05 → Non-stationary |

Using both tests together avoids false conclusions: if both agree, you're confident. If they disagree, you're in a borderline case requiring further investigation (structural breaks, near-unit-root).

In [ ]:
# Stationarity tests
def test_stationarity(series, name):
    """Perform ADF and KPSS tests"""
    # ADF Test
    adf_result = adfuller(series)
    adf_stat, adf_pvalue, _, _, adf_crit, _ = adf_result
    
    # KPSS Test
    kpss_result = kpss(series, regression='c')
    kpss_stat, kpss_pvalue, _, kpss_crit = kpss_result
    
    return {
        'Series': name,
        'ADF Stat': adf_stat,
        'ADF p-value': adf_pvalue,
        'KPSS Stat': kpss_stat,
        'KPSS p-value': kpss_pvalue,
        'Stationary (ADF)': 'Yes' if adf_pvalue < 0.05 else 'No',
        'Stationary (KPSS)': 'No' if kpss_pvalue < 0.05 else 'Yes'
    }

# Test each series
stationarity_results = []
for col in df_ts.columns:
    stationarity_results.append(test_stationarity(df_ts[col], col))

stationarity_df = pd.DataFrame(stationarity_results)
print("=" * 100)
print("STATIONARITY TEST RESULTS")
print("=" * 100)
print(stationarity_df.to_string(index=False))

# First differences to induce stationarity
df_diff = df_ts.diff().dropna()

print("\n" + "=" * 100)
print("AFTER FIRST DIFFERENCING")
print("=" * 100)
stationarity_diff = []
for col in df_diff.columns:
    stationarity_diff.append(test_stationarity(df_diff[col], col + ' (Differenced)'))

stationarity_diff_df = pd.DataFrame(stationarity_diff)
print(stationarity_diff_df.to_string(index=False))

# Visualize differences
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for idx, col in enumerate(df_ts.columns):
    ax_orig = axes.flat[idx]
    ax_orig.plot(df_ts.index, df_ts[col], label='Original', linewidth=1.5)
    ax_orig.set_title(f'{col} - Original vs Differenced', fontweight='bold')
    ax_orig.legend()
    ax_orig.grid(alpha=0.3)
    
    # Add second axis for differences
    ax_diff = ax_orig.twinx()
    ax_diff.plot(df_diff.index, df_diff[col], label='Differenced', color='red', linewidth=1, alpha=0.7)
    ax_diff.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 4. Autocorrelation Analysis (ACF & PACF)

Analyze autocorrelation and partial autocorrelation. ACF/PACF plots help identify ARIMA and GARCH orders.

### Why Do We Need ACF & PACF Analysis?

**The Core Problem:** Before fitting any model, you need to know what kind of memory the series has — does today's value depend on yesterday's shock only, or does it have a multi-period echo? ACF and PACF are the diagnostic tools that answer this.

**Practical Examples:**
- **ARIMA Model Order Selection**: ACF cuts off at lag 2 and PACF decays slowly → the series has MA(2) structure. Without ACF/PACF diagnostics, you would guess the order blindly, leading to an under/over-fitted model and poor forecasts.
- **Mean Reversion Signal Extraction**: A statistical arbitrage desk trading bond futures examines ACF of the basis spread. If ACF is strongly negative at lag 1 (anti-persistent), this confirms mean-reversion behavior and justifies a fade-the-move strategy.
- **High-Frequency Market Microstructure**: At millisecond resolution, bid-ask bounce creates strong negative autocorrelation in tick returns. HFT firms model this via MA structure to avoid trading against the bounce and improve execution quality.
- **Volatility Clustering Detection (ARCH Effects)**: The ACF of **squared residuals** (rather than returns themselves) tests for ARCH effects. Significant ACF in squared residuals confirms that volatility clusters — periods of high vol follow high vol — justifying GARCH model fitting.
- **Earnings Forecast Models**: Quarterly EPS time series for U.S. companies often show strong autocorrelation at lag 4 (seasonal quarterly pattern). ACF at lag 4 being significant tells analysts to use seasonal ARIMA (SARIMA) rather than vanilla ARIMA.

**Reading ACF/PACF Patterns:**
| ACF Pattern | PACF Pattern | Implied Model |
|------------|-------------|---------------|
| Cuts off at lag q | Gradually decays | MA(q) |
| Gradually decays | Cuts off at lag p | AR(p) |
| Both gradually decay | Both gradually decay | ARMA(p,q) |
| Spikes at seasonal lag s | Spikes at lag p | SARIMA |
| Squared residuals ACF significant | — | ARCH/GARCH effects present |

In [ ]:
# ACF/PACF for mean-reverting series (good example)
series_to_analyze = df_diff['mean_revert']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ACF
plot_acf(series_to_analyze, lags=40, ax=axes[0, 0], title='ACF - Mean-Revert (Differenced)')
axes[0, 0].set_xlabel('Lag')
axes[0, 0].set_ylabel('Autocorrelation')

# PACF
plot_pacf(series_to_analyze, lags=40, ax=axes[0, 1], title='PACF - Mean-Revert (Differenced)', method='ywm')
axes[0, 1].set_xlabel('Lag')
axes[0, 1].set_ylabel('Partial Autocorrelation')

# ACF of squared residuals (volatility clustering)
residuals = series_to_analyze - series_to_analyze.mean()
squared_residuals = residuals ** 2

plot_acf(squared_residuals, lags=40, ax=axes[1, 0], title='ACF - Squared Residuals')
axes[1, 0].set_xlabel('Lag')
axes[1, 0].set_ylabel('Autocorrelation')

# PACF of squared residuals
plot_pacf(squared_residuals, lags=40, ax=axes[1, 1], title='PACF - Squared Residuals', method='ywm')
axes[1, 1].set_xlabel('Lag')
axes[1, 1].set_ylabel('Partial Autocorrelation')

plt.tight_layout()
plt.show()

print("✓ Strong ACF in squared residuals suggests GARCH modeling is appropriate")

## 5. ARIMA Modeling

Fit ARIMA models to capture temporal dependencies. Auto-select optimal (p,d,q) parameters using information criteria.

### Why Do We Need ARIMA Modeling?

**The Core Problem:** Moving averages and linear trends cannot capture the complex serial dependencies in financial time series. ARIMA (AutoRegressive Integrated Moving Average) explicitly models these dependencies in a principled, testable framework.

**Practical Examples:**
- **Short-Term Interest Rate Forecasting**: The Fed Funds Rate is modeled with ARIMA by fixed income desks to forecast where rates will be in 3–6 months. An AR(1) component captures mean reversion toward a long-run level; the integrated (I) component handles the persistence seen in rate cycles.
- **Commodity Inventory Forecasting**: Natural gas storage levels (EIA weekly reports) drive large price moves. Energy traders fit ARIMA to storage change data to forecast next week's report before it's published, allowing pre-positioning ahead of the data release.
- **Credit Card Delinquency Forecasting**: Consumer banks use ARIMA to forecast monthly charge-off rates. The model captures the autocorrelation in delinquency trends, enabling more accurate loan-loss provisioning and capital planning.
- **Election Forecasting / Polling**: Political statisticians use ARIMA on time series of polling data to produce forecasts, correctly treating each poll as a noisy observation from an underlying autoregressive approval process.
- **Backtesting Signal Alpha Decay**: Quant researchers fit ARIMA on their "alpha signals" to understand how quickly their edge decays. A signal with strong AR(1) persistence decays slowly, justifying longer holding periods.

**ARIMA(p,d,q) Components:**
$$\underbrace{\phi(B)}_{\text{AR: p lags}} \underbrace{(1-B)^d}_{\text{Differencing: d times}} X_t = \underbrace{\theta(B)}_{\text{MA: q shocks}} \varepsilon_t$$

| Parameter | What It Captures | How to Identify |
|-----------|-----------------|----------------|
| p (AR order) | How many past values predict today | PACF cuts off at lag p |
| d (differencing) | Stationarity via differencing | ADF/KPSS test result |
| q (MA order) | How many past shocks influence today | ACF cuts off at lag q |

In [ ]:
# Select optimal ARIMA order
series = df_ts['mean_revert']

try:
    auto_order = arima_order_select_ic(series, max_p=5, max_d=2, max_q=5, ic='aic')
    best_order = auto_order['aic'][0]
    print(f"Auto-selected ARIMA order: {best_order}")
except:
    best_order = (1, 1, 1)
    print(f"Using fallback ARIMA order: {best_order}")

# Fit ARIMA model
arima_model = ARIMA(series, order=best_order)
arima_fit = arima_model.fit()

print("\n" + "=" * 80)
print("ARIMA MODEL SUMMARY")
print("=" * 80)
print(arima_fit.summary())

# In-sample fit and forecast
forecast_steps = 50
forecast = arima_fit.get_forecast(steps=forecast_steps)
forecast_ci = forecast.conf_int()
forecast_values = forecast.predicted_mean

# Plot fit and forecast
fig, ax = plt.subplots(figsize=(16, 7))

# Original data
ax.plot(series.index, series.values, label='Actual', linewidth=2, color='steelblue')

# Fitted values
ax.plot(series.index, arima_fit.fittedvalues, label='Fitted', linewidth=2, color='green', alpha=0.7)

# Forecast
forecast_idx = pd.date_range(start=series.index[-1], periods=forecast_steps+1, freq='D')[1:]
ax.plot(forecast_idx, forecast_values, label='Forecast', linewidth=2, color='red', linestyle='--')

# Confidence intervals
ax.fill_between(forecast_idx, 
                forecast_ci.iloc[:, 0], 
                forecast_ci.iloc[:, 1],
                alpha=0.2, color='red', label='95% CI')

ax.set_title(f'ARIMA{best_order} Model: Fit and Forecast', fontsize=12, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price')
ax.legend(loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Model diagnostics
arima_fit.plot_diagnostics(figsize=(14, 10))
plt.suptitle('ARIMA Model Diagnostics', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 6. GARCH Modeling for Volatility Forecasting

Fit GARCH (Generalized Autoregressive Conditional Heteroskedasticity) models to capture time-varying volatility.

### Why Do We Need GARCH Modeling?

**The Core Problem:** ARIMA models the conditional mean of returns, but the variance of returns is not constant — it clusters in high-volatility episodes and contracts in calm periods. GARCH explicitly models this time-varying variance, which is essential for risk management and options pricing.

**Practical Examples:**
- **Options Market Making**: Every market maker at a major derivatives desk uses GARCH to estimate near-term realized volatility. By comparing GARCH-implied volatility against the market's implied volatility (IV), traders identify when options are mispriced — buying cheap IV when GARCH forecasts suggest vol will spike.
- **Daily VaR Calculation**: Under the Basel Internal Models Approach, banks calculate 10-day 99% VaR using GARCH conditional volatility rather than constant historical volatility. When GARCH detects a volatility spike, VaR automatically increases and cash is held in reserve — preventing the bank from being caught undercapitalized.
- **Volatility Term Structure Trading**: The "vol surface" in options markets prices different expiries at different implied vols. GARCH mean-reversion parameters control how quickly volatility returns to its long-run average, providing a fundamental model for the term structure of volatility.
- **Risk Parity Rebalancing Trigger**: Risk parity portfolios like Bridgewater's All Weather use rolling GARCH estimates to scale positions. When GARCH vol doubles on equity, bond allocation increases proportionally to maintain equal risk contribution. This is done daily — requiring a fast, real-time vol model.
- **Flash Crash Risk**: On May 6, 2010, GARCH models tuned to recent low-vol conditions failed to forecast the extreme intraday spike. Firms using GARCH with regime-switching components (GJR-GARCH) had earlier warning of the volatility shift.

**GARCH(1,1) Model:**
$$\sigma_t^2 = \omega + \alpha \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

| Parameter | Meaning | Typical Value |
|-----------|---------|---------------|
| ω (omega) | Long-run variance floor | Small positive |
| α (alpha) | Reaction to news (ARCH effect) | 0.05 – 0.15 |
| β (beta) | Volatility persistence | 0.80 – 0.95 |
| α + β < 1 | Stationarity condition | Must hold |

High β means volatility is highly persistent — once it spikes, it stays elevated for days to weeks.

In [ ]:
# Calculate returns for GARCH modeling
returns = df_ts['mean_revert'].pct_change().dropna() * 100  # In percentage

# Fit GARCH(1,1) model
garch_model = arch_model(returns, vol='Garch', p=1, q=1)
garch_fit = garch_model.fit(disp='off')

print("=" * 80)
print("GARCH(1,1) MODEL SUMMARY")
print("=" * 80)
print(garch_fit.summary())

# Get conditional volatility
garch_volatility = garch_fit.conditional_volatility

# Forecast volatility
garch_forecast = garch_fit.forecast(horizon=50)
garch_volatility_forecast = garch_forecast.variance.iloc[-1, :].values ** 0.5

# Plot results
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Plot returns
ax = axes[0]
ax.plot(returns.index, returns.values, label='Returns (%)', linewidth=1, color='steelblue', alpha=0.7)
ax.set_title('Daily Returns', fontweight='bold')
ax.set_ylabel('Return (%)')
ax.grid(alpha=0.3)
ax.legend()

# Plot conditional volatility and forecast
ax = axes[1]
ax.plot(garch_volatility.index, garch_volatility.values, label='Conditional Volatility', 
        linewidth=1.5, color='red')

forecast_idx = pd.date_range(start=garch_volatility.index[-1], periods=51, freq='D')[1:]
ax.plot(forecast_idx, garch_volatility_forecast, label='Volatility Forecast', 
        linewidth=1.5, color='orange', linestyle='--')

ax.set_title('GARCH(1,1) Conditional Volatility and Forecast', fontweight='bold')
ax.set_ylabel('Volatility')
ax.set_xlabel('Date')
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nCurrent Volatility: {garch_volatility.iloc[-1]:.4f}%")
print(f"Avg Volatility (10-day): {garch_volatility.tail(10).mean():.4f}%")

## 7. Trend & Seasonality Decomposition

Decompose time series into trend, seasonality, and residual components using classical decomposition.

### Why Do We Need Trend & Seasonality Decomposition?

**The Core Problem:** If a time series has a strong trend or seasonal pattern, any model fitted on the raw series will conflate the trend/seasonal signal with the actual noise. Decomposition separates what is "expected structure" from what is a genuine anomaly.

**Practical Examples:**
- **Retail Sales Forecasting**: Amazon and Walmart forecast inventory weeks ahead. Raw monthly sales have massive December spikes and January drops. Without STL decomposition, an ARIMA model would treat every December spike as a "surprise" rather than a known seasonal pattern, severely overestimating forecast uncertainty.
- **Energy Market Trading**: Natural gas futures exhibit strong winter/summer seasonality. Gas traders decompose 5-year spot price histories to isolate the trend (supply/demand secular changes) from seasonal cycles, enabling them to price seasonal supply contracts accurately without conflating the two.
- **Economic Data Seasonality Adjustment**: The Bureau of Labor Statistics reports "seasonally adjusted" non-farm payrolls. The adjustment is a seasonal decomposition (Census X-13 method, related to STL) that removes predictable seasonal hiring patterns to reveal true economic signal in the jobs data that drives Fed rate decisions.
- **Anomaly Detection in Market Data**: After decomposing a financial series, the **residual** component represents unexplained variation. Residuals 3σ above/below their mean are anomalies: unexpected corporate events, data errors, or genuine market dislocations that require investigation.
- **Mean Reversion on Deseasonalized Spreads**: Commodity spread traders (crack spread in oil refining) decompose spreads to remove seasonal refinery maintenance patterns, then apply mean-reversion models only on the stationary residual component.

**STL Decomposition Formula:**
$$Y_t = T_t + S_t + R_t$$

Where:
- $T_t$ = Trend component (slow-moving, structural)
- $S_t$ = Seasonal component (predictable, repeating cycle)
- $R_t$ = Residual / Irregular component (genuinely unexplained variation)

In [ ]:
from statsmodels.tsa.seasonal import STL

# Use the seasonal series for decomposition (most interesting)
seasonal_series = data['seasonal']

# Apply STL decomposition (Seasonal and Trend decomposition using Loess)
# robust=True handles outliers better
stl = STL(seasonal_series, seasonal=13, trend=27, robust=True)
result = stl.fit()

# Plot decomposition
fig, axes = plt.subplots(4, 1, figsize=(14, 8))

# Original series
axes[0].plot(seasonal_series.index, seasonal_series.values, color='steelblue', linewidth=1.5)
axes[0].set_ylabel('Original', fontsize=10, fontweight='bold')
axes[0].set_title('STL Decomposition of Seasonal Time Series', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Trend component
axes[1].plot(result.trend.index, result.trend.values, color='darkgreen', linewidth=1.5)
axes[1].set_ylabel('Trend', fontsize=10, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Seasonal component
axes[2].plot(result.seasonal.index, result.seasonal.values, color='darkorange', linewidth=1.5)
axes[2].set_ylabel('Seasonal', fontsize=10, fontweight='bold')
axes[2].grid(True, alpha=0.3)

# Residual component
axes[3].plot(result.resid.index, result.resid.values, color='crimson', linewidth=1.5)
axes[3].set_ylabel('Residual', fontsize=10, fontweight='bold')
axes[3].set_xlabel('Time', fontsize=10, fontweight='bold')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print decomposition variance
var_original = seasonal_series.var()
var_trend = result.trend.var()
var_seasonal = result.seasonal.var()
var_residual = result.resid.var()

print(f"Original Series Variance: {var_original:.4f}")
print(f"Trend Variance: {var_trend:.4f} ({100*var_trend/var_original:.2f}%)")
print(f"Seasonal Variance: {var_seasonal:.4f} ({100*var_seasonal/var_original:.2f}%)")
print(f"Residual Variance: {var_residual:.4f} ({100*var_residual/var_original:.2f}%)")


## 8. Mean Reversion Analysis & Half-Life Estimation

Test for mean reversion and estimate the half-life of mean-reverting processes using Ornstein-Uhlenbeck framework.

### Why Do We Need Mean Reversion Analysis & Half-Life Estimation?

**The Core Problem:** Knowing that a series "tends to revert" is not enough to trade it. You need to know *how fast* it reverts — the half-life. A spread that takes 200 days to revert is untradable; one that takes 5 days is a high-frequency edge.

**Practical Examples:**
- **Pairs Trading Holding Period**: A quantitative equity analyst finds that the LVMH/Kering luxury goods spread has a 12-day half-life. This means the hedge fund holds the position for ~1 half-life (12 days) before expecting to capture 50% of the profit. This directly sets the strategy's turnover, transaction cost budget, and required Sharpe ratio.
- **Fixed Income Relative Value**: Bond traders use Ornstein-Uhlenbeck (OU) half-life to model the persistence of yield curve anomalies. A 2s10s yield spread with a 30-day half-life supports a carry trade; a 3-day half-life supports an intraday mean-reversion strategy.
- **FX Mean Reversion (Purchasing Power Parity)**: Academic research (and central bank FX desks) model exchange rates as mean-reverting toward PPP equilibrium, with typical half-lives of 3–5 years. This long half-life explains why short-term PPP-based FX trades don't work but long-term sovereign wealth fund allocation does.
- **Momentum vs. Mean Reversion Regime Detection**: The Hurst exponent from this analysis determines the current regime. Trend-following CTAs dynamically switch between trend and mean-reversion modes based on rolling Hurst estimates across asset classes.
- **Stat Arb Signal Expiry**: A quant running statistical arbitrage at a hedge fund sets signal expiry = 1 half-life. Signals that haven't converted to profit within 1 half-life are closed at market — cutting losers before the spread can drift further away.

**Ornstein-Uhlenbeck Process:**
$$dX_t = \theta(\mu - X_t)dt + \sigma\,dW_t$$

| Parameter | Meaning | Trading Implication |
|-----------|---------|---------------------|
| θ (speed) | Mean reversion rate | Higher θ = faster revert = shorter holding period |
| μ (mean) | Long-run equilibrium | Entry/exit signal reference point |
| Half-life | $\ln(2)/\theta$ | Average days to reach 50% of equilibrium |

$$\text{Half-Life} = \frac{\ln(2)}{\theta}$$

In [ ]:
from scipy import stats as sp_stats

# Calculate log-price and price changes
log_price = np.log(data['mean_revert'])
price_diff = log_price.diff().dropna()

# Test 1: Hurst Exponent (mean reversion indicator)
# H < 0.5 suggests mean reversion, H = 0.5 is random walk, H > 0.5 is trending
def calculate_hurst_exponent(timeseries, lags_range=None):
    """Calculate Hurst Exponent for mean reversion detection"""
    if lags_range is None:
        lags_range = range(2, 50)
    
    lags = []
    tau = []
    
    for lag in lags_range:
        # Calculate returns with lag
        delta = np.diff(timeseries, lag)
        # Calculate standard deviation of differences
        s_lag = np.sqrt(np.mean(delta**2))
        lags.append(lag)
        tau.append(s_lag)
    
    # Log-log regression
    tau = np.array(tau)
    lags = np.array(lags)
    poly = np.polyfit(np.log(lags), np.log(tau), 1)
    hurst = poly[0]
    
    return hurst, poly, lags, tau

hurst_exp, poly, lags, tau = calculate_hurst_exponent(log_price.values)

print(f"Hurst Exponent: {hurst_exp:.4f}")
print(f"Interpretation: {'Mean-Reverting' if hurst_exp < 0.5 else 'Trending' if hurst_exp > 0.5 else 'Random Walk'}")

# Test 2: Ornstein-Uhlenbeck Half-Life Estimation
# For mean-reverting process: dX_t = θ(μ - X_t)dt + σ dW_t
# Half-life = ln(2) / θ, where θ is mean reversion speed
def estimate_ornstein_uhlenbeck_halflife(timeseries):
    """Estimate half-life of mean reversion using OU model"""
    # Regress price change on current price level
    X = timeseries[:-1].values
    y = timeseries.diff().dropna().values
    
    # Add constant for intercept
    X_with_const = np.column_stack([np.ones(len(X)), X])
    
    # OLS regression: ΔX = α + β*X
    params, residuals, rank, s = np.linalg.lstsq(X_with_const, y, rcond=None)
    beta = params[1]
    
    # Half-life = -ln(2) / ln(1 + β)
    if beta < -1:
        half_life = np.nan
        print("Warning: Mean reversion coefficient suggests non-stationary process")
    else:
        half_life = -np.log(2) / np.log(1 + beta)
    
    return half_life, beta

half_life, beta = estimate_ornstein_uhlenbeck_halflife(log_price)

print(f"\nMean Reversion Speed (β): {beta:.6f}")
print(f"Half-Life (days): {half_life:.2f}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Plot 1: Mean-reverting series
axes[0, 0].plot(data.index, data['mean_revert'].values, color='steelblue', linewidth=1.5)
axes[0, 0].axhline(y=data['mean_revert'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[0, 0].fill_between(data.index, 
                        data['mean_revert'].mean() - data['mean_revert'].std(),
                        data['mean_revert'].mean() + data['mean_revert'].std(),
                        alpha=0.2, color='red', label='±1 Std')
axes[0, 0].set_title('Mean-Reverting Time Series', fontweight='bold')
axes[0, 0].set_ylabel('Value', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Hurst Exponent Log-Log
axes[0, 1].scatter(np.log(lags), np.log(tau), alpha=0.6, s=50, color='darkgreen')
fitted_line = np.poly1d(poly)(np.log(lags))
axes[0, 1].plot(np.log(lags), fitted_line, 'r--', linewidth=2, label=f'Hurst H={hurst_exp:.3f}')
axes[0, 1].set_xlabel('log(Lag)', fontweight='bold')
axes[0, 1].set_ylabel('log(Volatility)', fontweight='bold')
axes[0, 1].set_title('Hurst Exponent Calculation', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Autocorrelation decay
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(data['mean_revert'].values, lags=50, ax=axes[1, 0])
axes[1, 0].set_title('ACF: Mean-Reverting Series Decay', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Price level vs change (for OU regression)
axes[1, 1].scatter(log_price[:-1], log_price.diff().dropna(), alpha=0.5, s=30, color='purple')
x_line = np.array([log_price.min(), log_price.max()])
y_line = params[0] + params[1] * x_line
axes[1, 1].plot(x_line, y_line, 'r--', linewidth=2, label=f'β={beta:.6f}')
axes[1, 1].set_xlabel('Log Price Level', fontweight='bold')
axes[1, 1].set_ylabel('Price Change', fontweight='bold')
axes[1, 1].set_title('Ornstein-Uhlenbeck Regression', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 9. Co-Integration Analysis

Test for co-integration between multiple assets using Johansen test (foundation for pairs trading).

### Why Do We Need Co-Integration Analysis?

**The Core Problem:** Two non-stationary (drifting) price series cannot be traded on correlation alone — correlation breaks down constantly. Co-integration identifies pairs that share a long-run equilibrium, meaning their spread is stationary even though each series individually is not.

**Practical Examples:**
- **Pairs Trading in Equity Markets**: The classic example is Royal Dutch Shell and Shell Transport — two share classes of the same company, listed on different exchanges. Co-integration exists because arbitrageurs enforce the long-run pricing relationship. Whenever the spread widens (co-integration temporarily broken), quants short the expensive leg and buy the cheap leg, profiting as it converges.
- **Credit Default Swap vs. Bond Spread**: Investment-grade credit desks trade the basis between CDS spreads and bond spreads. Theoretically co-integrated (both price the same credit risk), basis widening during market dislocations (2008, 2020 COVID) creates convergence trades that returned 30–60% for those who knew the co-integration framework.
- **Gold vs. Mining ETF**: Gold bullion prices and gold mining stock ETFs (GDX) should be co-integrated via the fundamental relationship that mining profits depend on gold prices. Statistical arbitrage funds trade deviations from this co-integrating relationship continuously.
- **Currency Triangular Arbitrage**: EUR/USD, GBP/USD, and EUR/GBP are co-integrated by no-arbitrage constraints (log-linear relationship). FX quants use Johansen tests to verify the co-integrating rank (should be 2 for 3 series) and trade deviations algorithmically.
- **Portfolio Pair Construction at Scale**: Systematic funds run Johansen tests across every possible pair of ~500 large-cap stocks weekly to identify co-integrated pairs with acceptable half-lives and liquidity, then allocate capital proportionally to the number of active pairs.

**Co-Integration vs. Correlation:**
| Measure | What It Tests | Stable Over Time? | Trading Use |
|---------|--------------|------------------|-------------|
| Correlation | Linear co-movement at a point in time | No — breaks in crises | Marketing, not trading |
| Co-integration | Shared long-run stochastic trend | Yes — structural relationship | Real pairs trading |

**Johansen Test Result:**
- Rank = 0: No co-integrating relationship (don't trade the pair)
- Rank = 1: One co-integrating vector (classic pairs trade)
- Rank ≥ 2: Multiple relationships (portfolio of 3+ assets possible)

In [ ]:
from statsmodels.tsa.vector_ar.vecm import johansen

# Create paired assets - use two simulated price series that we'll make co-integrated
np.random.seed(42)
n = 252

# Create two potentially co-integrated series
# X1: random walk
X1 = np.cumsum(np.random.randn(n)) + 100

# X2: partially track X1 with added drift
# This creates co-integration if they share a common stochastic trend
X2 = 0.8 * X1 + np.cumsum(np.random.randn(n) * 0.5) + 20

# Store in DataFrame
coint_data = pd.DataFrame({
    'Asset_1': X1,
    'Asset_2': X2
}, index=data.index)

# Perform Johansen co-integration test
# This tests for the number of co-integrating relationships
result = johansen(coint_data, det_order=0, k_ar_diff=1)

# Extract test statistics
trace_stat = result.lr1[:, 0]  # Eigenvalue test statistic (Trace)
crit_90 = result.cvt[:, 0]     # 90% critical value
crit_95 = result.cvt[:, 1]     # 95% critical value
crit_99 = result.cvt[:, 2]     # 99% critical value

print("Johansen Co-integration Test Results")
print("=" * 60)
print(f"{'Rank':<6} {'Stat':<10} {'90%':<10} {'95%':<10} {'99%':<10} {'Coint?':<8}")
print("-" * 60)

is_cointegrated = False
for i in range(len(trace_stat)):
    is_coint = "Yes" if trace_stat[i] > crit_95[i] else "No"
    if trace_stat[i] > crit_95[i]:
        is_cointegrated = True
    print(f"{i:<6} {trace_stat[i]:<10.4f} {crit_90[i]:<10.4f} {crit_95[i]:<10.4f} {crit_99[i]:<10.4f} {is_coint:<8}")

print(f"\nCo-integrated pair: {'YES' if is_cointegrated else 'NO'}")

# If co-integrated, calculate the spread (co-integrating combination)
# Spread = Asset_1 - β*Asset_2, where β is estimated from co-integration
if is_cointegrated:
    # Simple spread: just the difference
    spread = coint_data['Asset_1'] - coint_data['Asset_2']
    
    # Test spread stationarity with ADF
    from statsmodels.tsa.stattools import adfuller
    adf_result = adfuller(spread)
    print(f"\nSpread ADF test p-value: {adf_result[1]:.6f}")
    print(f"Spread is stationary: {'YES' if adf_result[1] < 0.05 else 'NO'}")
    
    # Generate trading signals based on spread
    spread_mean = spread.mean()
    spread_std = spread.std()
    
    # Signals: buy when spread is low (below mean-std), sell when high (above mean+std)
    signals = np.zeros(len(spread))
    signals[spread < (spread_mean - spread_std)] = 1      # Buy signal
    signals[spread > (spread_mean + spread_std)] = -1     # Sell signal
    
    print(f"\nTrading Signal Summary:")
    print(f"Buy signals (spread low): {np.sum(signals == 1)}")
    print(f"Sell signals (spread high): {np.sum(signals == -1)}")
    print(f"Neutral: {np.sum(signals == 0)}")
    
    # Visualization
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # Plot 1: Asset prices
    axes[0].plot(coint_data.index, coint_data['Asset_1'], label='Asset 1', color='blue', linewidth=1.5)
    axes[0].plot(coint_data.index, coint_data['Asset_2'], label='Asset 2', color='green', linewidth=1.5)
    axes[0].set_title('Co-integrated Asset Pairs', fontweight='bold', fontsize=12)
    axes[0].set_ylabel('Price', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Spread
    axes[1].plot(coint_data.index, spread, color='steelblue', linewidth=1.5)
    axes[1].axhline(y=spread_mean, color='red', linestyle='--', linewidth=2, label='Mean')
    axes[1].axhline(y=spread_mean + spread_std, color='orange', linestyle=':', linewidth=2, label='+1σ')
    axes[1].axhline(y=spread_mean - spread_std, color='orange', linestyle=':', linewidth=2, label='-1σ')
    axes[1].fill_between(coint_data.index, spread_mean - spread_std, spread_mean + spread_std, 
                         alpha=0.2, color='orange')
    axes[1].set_title('Co-integrating Spread (Asset 1 - Asset 2)', fontweight='bold', fontsize=12)
    axes[1].set_ylabel('Spread', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Trading signals
    colors = ['red' if s < 0 else 'green' if s > 0 else 'gray' for s in signals]
    axes[2].scatter(coint_data.index, signals, c=colors, alpha=0.6, s=30)
    axes[2].plot(coint_data.index, spread / spread_std, color='steelblue', linewidth=1, alpha=0.7, label='Normalized Spread')
    axes[2].set_title('Trading Signals from Spread', fontweight='bold', fontsize=12)
    axes[2].set_ylabel('Signal', fontweight='bold')
    axes[2].set_xlabel('Time', fontweight='bold')
    axes[2].set_xlim(coint_data.index[0], coint_data.index[-1])
    axes[2].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[2].legend(['Spread (σ-normalized)', 'Buy signals', 'Sell signals'])
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("\nAssets are not co-integrated; pairs trading strategy may not be suitable.")


## 10. Mean Reversion Trading Strategy

Implement a simple mean reversion trading strategy using Bollinger Bands and evaluate performance metrics.

### Why Do We Need a Mean Reversion Trading Strategy?

**The Core Problem:** Statistical analysis alone doesn't generate profit — you need a formalized, rules-based strategy that translates statistical observations into trade entries, exits, position sizing, and performance measurement. Without this step, the analysis remains academic.

**Practical Examples:**
- **Renaissance Technologies Medallion Fund**: The most profitable hedge fund in history (avg 66% gross return 1988–2018) is primarily a statistical mean-reversion strategy. Bollinger-Band-style entry/exit signals on hundreds of assets simultaneously are the conceptual foundation, enhanced by high-frequency data and advanced signal processing.
- **Crypto Market Making**: On decentralized exchanges, market makers use Bollinger Bands on the price of a token relative to its moving average to dynamically adjust their bid-ask spread. When price is at the upper band, they widen the ask (expecting reversion); at the lower band, they widen the bid.
- **Systematic Fixed Income Trading**: Interest rate spread desks at J.P. Morgan, Goldman, and Barclays run Bollinger-Band-type strategies on yield curve spreads (2s10s, 5s30s). When the spread breaches 2 standard deviations from its 60-day mean, a trade is initiated with a pre-defined risk limit.
- **ETF Statistical Arbitrage**: The premium/discount of an ETF to its NAV (Net Asset Value) is a mean-reverting spread. Authorized participants and arb desks trade whenever the Bollinger Band level is breached, keeping ETF prices tethered to their NAV.

**Performance Metrics Decoded:**
| Metric | What It Measures | Good Value |
|--------|-----------------|-----------|
| Total Return | Absolute profit over period | Context-dependent |
| Sharpe Ratio | Return per unit of total risk | > 1.0 (good), > 2.0 (excellent) |
| Sortino Ratio | Return per unit of *downside* risk only | Higher is better than Sharpe alone |
| Max Drawdown | Worst peak-to-trough loss | < 20% (institutional), < 10% (conservative) |
| Win Rate | % of profitable days/trades | 50–55% (mean reversion is typical) |

**Why Sharpe Alone Is Insufficient:**
A strategy can have Sharpe = 1.5 but a −40% max drawdown (common in momentum strategies). The Sortino ratio and drawdown together reveal the *experience* of running the strategy — something Sharpe masks.

In [ ]:
# Use mean-reverting series and create trading strategy
price_series = data['mean_revert'].copy()

# Calculate SMA (Simple Moving Average)
sma_period = 20
sma = price_series.rolling(window=sma_period).mean()

# Calculate Bollinger Bands
bb_period = 20
bb_std = 2
rolling_mean = price_series.rolling(window=bb_period).mean()
rolling_std = price_series.rolling(window=bb_period).std()
upper_band = rolling_mean + (rolling_std * bb_std)
lower_band = rolling_mean - (rolling_std * bb_std)

# Trading signals: -1 (short when price > upper), 0 (hold), 1 (long when price < lower)
signals = np.zeros(len(price_series))
signals[price_series > upper_band] = -1  # Oversold (short)
signals[price_series < lower_band] = 1   # Oversold (long)

# Position changes (to identify entry/exit points)
positions = signals.copy()
positions[np.isnan(upper_band)] = 0      # No signal during warmup

# Calculate returns from trading strategy
# Simple approach: holdings follow signals
daily_returns = price_series.pct_change()
strategy_returns = daily_returns * positions.shift(1)  # Use yesterday's signal

# Calculate cumulative returns
cum_market_returns = (1 + daily_returns).cumprod() - 1
cum_strategy_returns = (1 + strategy_returns).cumprod() - 1

# Performance metrics
total_return_market = cum_market_returns.iloc[-1]
total_return_strategy = cum_strategy_returns.iloc[-1]

# Sharpe Ratio (annualized)
sharpe_market = np.mean(daily_returns) / np.std(daily_returns) * np.sqrt(252)
sharpe_strategy = np.mean(strategy_returns) / np.std(strategy_returns) * np.sqrt(252)

# Sortino Ratio (downside deviation)
downside_returns_market = daily_returns[daily_returns < 0]
downside_returns_strategy = strategy_returns[strategy_returns < 0]
sortino_market = np.mean(daily_returns) / np.std(downside_returns_market) * np.sqrt(252)
sortino_strategy = np.mean(strategy_returns) / np.std(downside_returns_strategy) * np.sqrt(252)

# Max Drawdown
cum_max_market = cum_market_returns.expanding().max()
drawdown_market = (cum_market_returns - cum_max_market) / (1 + cum_max_market)
max_dd_market = drawdown_market.min()

cum_max_strategy = cum_strategy_returns.expanding().max()
drawdown_strategy = (cum_strategy_returns - cum_max_strategy) / (1 + cum_max_strategy)
max_dd_strategy = drawdown_strategy.min()

# Win rate
winning_days_strategy = np.sum(strategy_returns > 0) if len(strategy_returns) > 0 else 0
total_days_strategy = np.sum(strategy_returns != 0) if len(strategy_returns) > 0 else 1
win_rate = winning_days_strategy / total_days_strategy if total_days_strategy > 0 else 0

# Print performance metrics
print("="*70)
print("MEAN REVERSION TRADING STRATEGY PERFORMANCE")
print("="*70)
print(f"\n{'Metric':<30} {'Buy & Hold':<20} {'Strategy':<20}")
print("-"*70)
print(f"{'Total Return':<30} {total_return_market:>18.2%} {total_return_strategy:>18.2%}")
print(f"{'Sharpe Ratio (annualized)':<30} {sharpe_market:>18.4f} {sharpe_strategy:>18.4f}")
print(f"{'Sortino Ratio (annualized)':<30} {sortino_market:>18.4f} {sortino_strategy:>18.4f}")
print(f"{'Max Drawdown':<30} {max_dd_market:>18.2%} {max_dd_strategy:>18.2%}")
print(f"{'Win Rate':<30} {'N/A':>20} {win_rate:>18.2%}")
print("-"*70)

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Plot 1: Price with Bollinger Bands and signals
axes[0].plot(price_series.index, price_series.values, label='Price', color='black', linewidth=2)
axes[0].plot(rolling_mean.index, rolling_mean.values, label='SMA(20)', color='blue', linewidth=1.5)
axes[0].fill_between(upper_band.index, upper_band.values, lower_band.values, 
                     alpha=0.2, color='gray', label='Bollinger Bands (σ=2)')
axes[0].plot(upper_band.index, upper_band.values, color='red', linestyle='--', linewidth=1, alpha=0.7)
axes[0].plot(lower_band.index, lower_band.values, color='green', linestyle='--', linewidth=1, alpha=0.7)

# Highlight buy/sell signals
buy_signals = signals == 1
sell_signals = signals == -1
axes[0].scatter(price_series.index[buy_signals], price_series.values[buy_signals], 
               color='green', marker='^', s=100, label='Buy Signal', zorder=5)
axes[0].scatter(price_series.index[sell_signals], price_series.values[sell_signals], 
               color='red', marker='v', s=100, label='Sell Signal', zorder=5)

axes[0].set_title('Price with Bollinger Bands & Trading Signals', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Price', fontweight='bold')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Plot 2: Cumulative returns comparison
axes[1].plot(cum_market_returns.index, cum_market_returns.values * 100, 
            label='Buy & Hold', color='blue', linewidth=2)
axes[1].plot(cum_strategy_returns.index, cum_strategy_returns.values * 100, 
            label='Mean Reversion Strategy', color='green', linewidth=2)
axes[1].set_title('Cumulative Returns: Strategy vs Buy & Hold', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Return (%)', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Drawdown comparison
axes[2].fill_between(drawdown_market.index, drawdown_market.values * 100, 0, 
                     alpha=0.5, color='blue', label='Buy & Hold Drawdown')
axes[2].fill_between(drawdown_strategy.index, drawdown_strategy.values * 100, 0, 
                     alpha=0.5, color='green', label='Strategy Drawdown')
axes[2].set_title('Drawdown Comparison', fontweight='bold', fontsize=12)
axes[2].set_ylabel('Drawdown (%)', fontweight='bold')
axes[2].set_xlabel('Time', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 11. Summary & Key Takeaways

Key concepts for quantitative time series management and practical applications in trading.

### Core Concepts Covered

1. **Stationarity Testing**: Used ADF and KPSS tests to differentiate between stationary and non-stationary time series
   - ADF null: Series has unit root (non-stationary)
   - KPSS null: Series is stationary
   - Differencing converts integrated series to stationary

2. **Autoregressive Processes**: 
   - ACF/PACF patterns identify AR/MA orders
   - Volatility clustering detected through squared residuals
   - ARIMA models combine autoregressive and differencing components

3. **Conditional Volatility**:
   - GARCH captures time-varying volatility (heteroskedasticity)
   - Volatility forecasts essential for options pricing and risk management
   - Conditional vs. unconditional volatility implications

4. **Trend & Seasonality**:
   - STL decomposition isolates trend, seasonal, and residual components
   - Useful for forecasting after removing structural patterns
   - Variance explained by each component reveals dominant dynamics

5. **Mean Reversion**:
   - Hurst exponent: H<0.5 suggests mean reversion, H>0.5 suggests trending behavior
   - Half-life measures speed of mean reversion (trading signal importance)
   - OU process framework enables mathematical modeling

6. **Co-Integration**:
   - Johansen test identifies stationary linear combinations of non-stationary series
   - Co-integrated pairs suitable for pairs/statistical arbitrage trading
   - Spread regression creates tradable zero-cost portfolio

7. **Strategy Implementation**:
   - Bollinger Bands provide entry/exit signals for mean reversion strategies
   - Performance metrics: Sharpe, Sortino, Max Drawdown, Win Rate
   - Backtesting reveals strategy effectiveness vs. buy-and-hold

### Practical Applications

- **Portfolio Risk Management**: Volatility forecasting with GARCH for VaR calculation
- **Pairs Trading**: Co-integration analysis to identify hedge pairs
- **Mean Reversion Trading**: Identify oversold/overbought conditions
- **Forecasting**: ARIMA for point estimates, GARCH for uncertainty bands
- **Anomaly Detection**: STL residuals reveal unexpected market movements

### Next Steps

- Integrate multiple strategies into portfolio optimization
- Backtest on historical data with realistic transaction costs
- Extend to multivariate models (Vector AR for multiple assets)
- Implement machine learning for parameter optimization
- Deploy strategies with proper risk controls and position sizing